# 01. Introducción a LLM Agents

**Nivel:** 🟢 Principiante  
**Tiempo estimado:** 60 minutos  
**Prerequisitos:** Familiaridad básica con LLMs y APIs

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Explicar la diferencia fundamental entre un LLM y un Agente basado en LLM
- Identificar los componentes clave de un agente (percepción, razonamiento, acción)
- Implementar un loop de agente simple desde cero
- Comprender el ciclo observación → pensamiento → acción → observación
- Configurar y usar APIs de LLMs (OpenAI, Anthropic, o modelos locales)

## 1. Motivación: ¿Por qué Agentes?

### El Problema con LLMs Puros

Imagina que le preguntas a ChatGPT: *"¿Cuál es el clima en Madrid ahora mismo?"*

El LLM responderá algo como: *"Lo siento, no tengo acceso a información en tiempo real..."* 

**¿Por qué?** Porque un LLM puro:
- Solo tiene conocimiento hasta su fecha de entrenamiento
- No puede acceder a APIs o herramientas externas
- No puede ejecutar acciones en el mundo real
- Solo puede generar texto basado en su contexto

### La Solución: Agentes

Un **Agente basado en LLM** puede:
1. **Razonar** sobre qué información necesita
2. **Decidir** usar una herramienta (ej: API de clima)
3. **Ejecutar** la acción (llamar a la API)
4. **Observar** el resultado
5. **Razonar nuevamente** con la nueva información
6. **Responder** al usuario con datos actualizados

### Pregunta Guía

**Al final de este notebook responderemos:**
*¿Cómo podemos transformar un LLM pasivo (solo genera texto) en un agente activo (razona y ejecuta acciones)?*

## 2. Intuición Visual: Anatomía de un Agente

### LLM vs Agente: Comparación

```
┌─────────────────────────────────────────────────────────────┐
│                      LLM (Pasivo)                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Input (Prompt) ──────► [LLM] ──────► Output (Text)       │
│                                                             │
│  • Solo procesa texto                                      │
│  • Una pasada, sin iteración                               │
│  • No puede usar herramientas                              │
│  • Conocimiento estático                                   │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    Agente (Activo)                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│           ┌──────────────────────────┐                     │
│           │                          │                     │
│           ▼                          │                     │
│  Observación ──► Razonamiento ──► Acción                  │
│      ▲         (LLM Core)        │                        │
│      │                           │                        │
│      │                           ▼                        │
│      │                      [Herramientas]                │
│      │                      • Web Search                  │
│      │                      • Calculator                  │
│      │                      • Database                    │
│      │                      • Code Exec                   │
│      └──────────────────────────┘                         │
│                                                             │
│  • Loop iterativo                                          │
│  • Usa herramientas externas                               │
│  • Actualiza conocimiento dinámicamente                    │
│  • Toma decisiones y ejecuta acciones                      │
└─────────────────────────────────────────────────────────────┘
```

### Componentes de un Agente

1. **Cerebro (LLM)**: Razona sobre qué hacer
2. **Memoria**: Recuerda interacciones pasadas y contexto
3. **Herramientas**: Capacidades extendidas (APIs, calculadora, etc.)
4. **Control Loop**: Orquesta el ciclo observar-pensar-actuar
5. **Planner** (opcional): Descompone tareas complejas en pasos

Visualizaremos esto con código a continuación.

In [ ]:
# Instalación de dependencias necesarias
# Descomenta si no las tienes instaladas

# !pip install openai anthropic python-dotenv plotly pandas
# Para modelos locales (opcional):
# !pip install transformers torch

In [ ]:
import os
import json
from typing import List, Dict, Optional, Callable
from dataclasses import dataclass
from datetime import datetime

# Para visualizaciones
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Para APIs de LLMs
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible. Instala con: pip install openai")

try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("⚠️  Anthropic no disponible. Instala con: pip install anthropic")

# Configuración
from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas correctamente")

## 3. Fundamentos Matemáticos: Formalización del Agente

### Definición Formal

Un **agente** puede formalizarse como una función que mapea secuencias de percepciones a acciones:

$$
\begin{align}
\pi: \mathcal{O}^* &\to \mathcal{A} \tag{1} \\
\text{donde: } & \\
\mathcal{O} &: \text{espacio de observaciones} \\
\mathcal{A} &: \text{espacio de acciones} \\
\pi &: \text{política del agente (la "estrategia")} \\
\mathcal{O}^* &: \text{historial de observaciones}
\end{align}
$$

### El Loop del Agente

En cada paso temporal $t$:

$$
\begin{align}
o_t &= \text{observe}(\text{environment}) \tag{2} \\
s_t &= \text{update\_state}(s_{t-1}, o_t) \tag{3} \\
a_t &= \pi(s_t) \tag{4} \\
\text{environment} &= \text{execute}(a_t) \tag{5}
\end{align}
$$

Donde:
- $o_t$: Observación en tiempo $t$
- $s_t$: Estado interno del agente (memoria)
- $a_t$: Acción elegida por la política
- $\pi$: Política implementada por el LLM

### Para LLM Agents

La política $\pi$ es implementada por un LLM:

$$
\begin{align}
\pi(s_t) &= \text{LLM}(\text{prompt}(s_t)) \tag{6} \\
\text{prompt}(s_t) &= \text{template}(\text{system\_msg}, \text{history}_t, \text{tools}) \tag{7}
\end{align}
$$

**Key Insight:**
> La "inteligencia" del agente emerge de cómo construimos el prompt que alimenta al LLM. El diseño del prompt determina qué tan bien razona y actúa el agente.

### Ejemplo Numérico

Supongamos:
- Usuario pregunta: "¿Cuánto es 15% de 340?"
- Agente tiene herramienta: `calculator`

**Paso 1:** $o_0$ = "¿Cuánto es 15% de 340?"  
**Paso 2:** LLM razona → decide usar `calculator(0.15 * 340)`  
**Paso 3:** Herramienta ejecuta → $o_1$ = "51.0"  
**Paso 4:** LLM genera respuesta final → "El 15% de 340 es 51"

Este loop puede iterar múltiples veces según la complejidad de la tarea.

## 4. Implementación Desde Cero: Simple Agent Loop

Implementaremos un agente básico con tres componentes:
1. **LLM Backend** (OpenAI, Anthropic, o simulado)
2. **Herramientas simples** (calculadora, hora actual)
3. **Loop de control**

In [ ]:
@dataclass
class AgentAction:
    """Representa una acción del agente"""
    tool: str  # Nombre de la herramienta a usar
    tool_input: str  # Input para la herramienta
    reasoning: str  # Razonamiento del agente

@dataclass
class Observation:
    """Representa una observación del entorno"""
    content: str  # Contenido de la observación
    timestamp: datetime  # Cuándo ocurrió

class SimpleTool:
    """Clase base para herramientas del agente"""
    def __init__(self, name: str, description: str, func: Callable):
        self.name = name
        self.description = description
        self.func = func
    
    def run(self, input_str: str) -> str:
        """Ejecuta la herramienta con el input dado"""
        try:
            result = self.func(input_str)
            return str(result)
        except Exception as e:
            return f"Error: {str(e)}"

# Definir herramientas simples
def calculator(expression: str) -> float:
    """Calcula expresiones matemáticas simples"""
    # ADVERTENCIA: eval es peligroso en producción, usar solo para demos
    # En producción, usa un parser matemático seguro
    return eval(expression)

def get_current_time(timezone: str = "UTC") -> str:
    """Retorna la hora actual"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Crear herramientas
calculator_tool = SimpleTool(
    name="calculator",
    description="Útil para hacer cálculos matemáticos. Input: expresión matemática en Python.",
    func=calculator
)

time_tool = SimpleTool(
    name="get_time",
    description="Retorna la fecha y hora actual.",
    func=get_current_time
)

print("✅ Herramientas creadas:", [calculator_tool.name, time_tool.name])

In [ ]:
class SimpleAgent:
    """
    Implementación básica de un agente LLM.
    
    El agente:
    1. Recibe una query del usuario
    2. Decide si necesita usar una herramienta
    3. Ejecuta la herramienta si es necesario
    4. Genera una respuesta final
    """
    
    def __init__(self, tools: List[SimpleTool], llm_backend: str = "simulated"):
        """
        Args:
            tools: Lista de herramientas disponibles
            llm_backend: "openai", "anthropic", o "simulated"
        """
        self.tools = {tool.name: tool for tool in tools}
        self.llm_backend = llm_backend
        self.history: List[Dict] = []  # Historial de interacciones
        
        # Configurar cliente LLM
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            self.model = "gpt-4-turbo-preview"
        elif llm_backend == "anthropic" and ANTHROPIC_AVAILABLE:
            self.client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
            self.model = "claude-3-opus-20240229"
        else:
            self.client = None
            print("⚠️  Usando LLM simulado para demostración")
    
    def _build_prompt(self, query: str) -> str:
        """Construye el prompt para el LLM"""
        tools_desc = "\n".join([
            f"- {name}: {tool.description}" 
            for name, tool in self.tools.items()
        ])
        
        prompt = f"""Eres un asistente útil que puede usar herramientas para responder preguntas.

Herramientas disponibles:
{tools_desc}

Para usar una herramienta, responde EXACTAMENTE en este formato:
RAZONAMIENTO: [tu razonamiento]
HERRAMIENTA: [nombre_herramienta]
INPUT: [input para la herramienta]

Si no necesitas una herramienta, responde directamente.

Pregunta del usuario: {query}

Tu respuesta:"""
        return prompt
    
    def _call_llm(self, prompt: str) -> str:
        """Llama al LLM y retorna la respuesta"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message.content
        
        elif self.llm_backend == "anthropic" and self.client:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        
        else:
            # LLM simulado para demostración
            return self._simulated_llm(prompt)
    
    def _simulated_llm(self, prompt: str) -> str:
        """Simula un LLM con reglas básicas (solo para demo)"""
        query_lower = prompt.lower()
        
        # Detectar si necesita calculadora
        if any(word in query_lower for word in ['calcular', 'cuánto es', '+', '-', '*', '/', '%']):
            # Intentar extraer expresión matemática
            import re
            # Buscar patrones como "15% de 340" o "2 + 2"
            match = re.search(r'(\d+)%\s+de\s+(\d+)', query_lower)
            if match:
                expr = f"{match.group(1)} * {match.group(2)} / 100"
            else:
                # Buscar expresiones matemáticas simples
                expr_match = re.search(r'[\d+\-*/().\s]+', query_lower)
                expr = expr_match.group(0) if expr_match else "1+1"
            
            return f"""RAZONAMIENTO: Necesito calcular una expresión matemática
HERRAMIENTA: calculator
INPUT: {expr}"""
        
        # Detectar si pregunta por hora
        elif any(word in query_lower for word in ['hora', 'fecha', 'tiempo']):
            return """RAZONAMIENTO: Usuario pregunta por la hora actual
HERRAMIENTA: get_time
INPUT: UTC"""
        
        # Respuesta directa
        return "Hola, puedo ayudarte con cálculos y consultar la hora."
    
    def _parse_action(self, llm_response: str) -> Optional[AgentAction]:
        """Parsea la respuesta del LLM para extraer una acción"""
        if "HERRAMIENTA:" not in llm_response:
            return None
        
        lines = llm_response.strip().split('\n')
        action_dict = {}
        
        for line in lines:
            if line.startswith("RAZONAMIENTO:"):
                action_dict['reasoning'] = line.replace("RAZONAMIENTO:", "").strip()
            elif line.startswith("HERRAMIENTA:"):
                action_dict['tool'] = line.replace("HERRAMIENTA:", "").strip()
            elif line.startswith("INPUT:"):
                action_dict['tool_input'] = line.replace("INPUT:", "").strip()
        
        if 'tool' in action_dict:
            return AgentAction(
                tool=action_dict.get('tool', ''),
                tool_input=action_dict.get('tool_input', ''),
                reasoning=action_dict.get('reasoning', '')
            )
        return None
    
    def run(self, query: str, max_iterations: int = 3, verbose: bool = True) -> str:
        """
        Ejecuta el loop del agente.
        
        Args:
            query: Pregunta del usuario
            max_iterations: Máximo número de iteraciones del loop
            verbose: Si True, imprime pasos intermedios
        
        Returns:
            Respuesta final del agente
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"🤖 AGENTE EJECUTÁNDOSE")
            print(f"{'='*60}")
            print(f"\n📝 Query: {query}\n")
        
        for iteration in range(max_iterations):
            if verbose:
                print(f"\n--- Iteración {iteration + 1} ---")
            
            # 1. Construir prompt y llamar LLM
            prompt = self._build_prompt(query)
            llm_response = self._call_llm(prompt)
            
            if verbose:
                print(f"\n💭 LLM Response:\n{llm_response}")
            
            # 2. Parsear acción
            action = self._parse_action(llm_response)
            
            # 3. Si no hay acción, retornar respuesta
            if action is None:
                if verbose:
                    print(f"\n✅ Respuesta final (sin herramientas)")
                return llm_response
            
            # 4. Ejecutar herramienta
            if action.tool not in self.tools:
                error_msg = f"Error: Herramienta '{action.tool}' no disponible"
                if verbose:
                    print(f"\n❌ {error_msg}")
                return error_msg
            
            if verbose:
                print(f"\n🔧 Usando herramienta: {action.tool}")
                print(f"📥 Input: {action.tool_input}")
            
            observation = self.tools[action.tool].run(action.tool_input)
            
            if verbose:
                print(f"📤 Output: {observation}")
            
            # 5. Actualizar query con observación
            query = f"""Pregunta original: {query}
Razonamiento: {action.reasoning}
Herramienta usada: {action.tool}
Resultado: {observation}

Por favor, proporciona la respuesta final al usuario basándote en esta información."""
        
        # Si llegamos al máximo de iteraciones
        final_prompt = self._build_prompt(query)
        final_response = self._call_llm(final_prompt)
        
        if verbose:
            print(f"\n✅ Respuesta final:\n{final_response}")
        
        return final_response

print("✅ Clase SimpleAgent implementada")

### Probemos nuestro agente

In [ ]:
# Crear agente con herramientas
agent = SimpleAgent(
    tools=[calculator_tool, time_tool],
    llm_backend="simulated"  # Cambiar a "openai" o "anthropic" si tienes API keys
)

# Probar con una pregunta matemática
result1 = agent.run("¿Cuánto es el 15% de 340?")
print(f"\n{'='*60}")
print(f"Resultado Final: {result1}")

In [ ]:
# Probar con pregunta de tiempo
result2 = agent.run("¿Qué hora es?")
print(f"\n{'='*60}")
print(f"Resultado Final: {result2}")

## 5. Versión con Framework: LangChain

Ahora veamos cómo hacer lo mismo con LangChain, un framework popular para construir aplicaciones con LLMs.

In [ ]:
# Instalar LangChain si no está disponible
# !pip install langchain langchain-openai langchain-community

try:
    from langchain.agents import initialize_agent, Tool, AgentType
    from langchain.llms import OpenAI as LangChainOpenAI
    from langchain_openai import ChatOpenAI
    LANGCHAIN_AVAILABLE = True
except ImportError:
    LANGCHAIN_AVAILABLE = False
    print("⚠️  LangChain no disponible. Instala con: pip install langchain langchain-openai")

if LANGCHAIN_AVAILABLE and OPENAI_AVAILABLE:
    # Definir herramientas para LangChain
    tools_langchain = [
        Tool(
            name="Calculator",
            func=lambda x: str(eval(x)),
            description="Útil para hacer cálculos matemáticos. Input debe ser una expresión válida de Python."
        ),
        Tool(
            name="Time",
            func=lambda x: datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            description="Retorna la fecha y hora actual."
        )
    ]
    
    # Inicializar LLM
    llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
    
    # Inicializar agente
    langchain_agent = initialize_agent(
        tools=tools_langchain,
        llm=llm,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        verbose=True
    )
    
    print("✅ Agente LangChain creado")
    
    # Probar
    # result = langchain_agent.run("¿Cuánto es 15% de 340?")
    # print(f"\nResultado LangChain: {result}")
else:
    print("⚠️  Saltando ejemplo de LangChain (dependencias no disponibles)")

### Comparación: Desde Cero vs Framework

| Aspecto | Desde Cero | LangChain |
|---------|------------|------------|
| **Control** | ✅ Total | ⚠️ Limitado |
| **Complejidad** | Alta (más código) | Baja (abstracción) |
| **Flexibilidad** | ✅ Máxima | ⚠️ Depende del framework |
| **Debugging** | Más difícil | Más fácil (logging built-in) |
| **Producción** | Requiere más trabajo | Listo para producción |
| **Aprendizaje** | ✅ Mejor comprensión | Más rápido de usar |
| **Herramientas** | Debes implementar todo | Ecosistema rico |

**Recomendación:**
- **Aprendizaje**: Implementa desde cero primero
- **Producción**: Usa frameworks (LangChain, LlamaIndex, etc.)
- **Proyectos complejos**: Combina ambos enfoques

## 6. Visualización del Loop del Agente

Creemos una visualización interactiva del proceso de razonamiento del agente.

In [ ]:
def visualize_agent_trace(steps: List[Dict]):
    """
    Visualiza el trace de ejecución de un agente.
    
    Args:
        steps: Lista de pasos, cada uno con {type, content, timestamp}
    """
    fig = go.Figure()
    
    # Colores para diferentes tipos de pasos
    colors = {
        'query': '#3498db',
        'reasoning': '#9b59b6',
        'tool': '#e74c3c',
        'observation': '#2ecc71',
        'response': '#f39c12'
    }
    
    # Crear timeline
    for i, step in enumerate(steps):
        fig.add_trace(go.Scatter(
            x=[i],
            y=[0],
            mode='markers+text',
            marker=dict(
                size=40,
                color=colors.get(step['type'], '#95a5a6'),
                line=dict(width=2, color='white')
            ),
            text=step['type'].upper(),
            textposition="top center",
            hovertext=step['content'],
            hoverinfo='text',
            showlegend=False
        ))
        
        # Conectar pasos con flechas
        if i < len(steps) - 1:
            fig.add_annotation(
                x=i+0.5, y=0,
                ax=i, ay=0,
                xref='x', yref='y',
                axref='x', ayref='y',
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor='#34495e'
            )
    
    fig.update_layout(
        title="Agent Execution Trace",
        xaxis=dict(showticklabels=False, showgrid=False),
        yaxis=dict(showticklabels=False, showgrid=False, range=[-1, 1]),
        height=300,
        template='plotly_white',
        hovermode='closest'
    )
    
    return fig

# Ejemplo de trace
example_trace = [
    {'type': 'query', 'content': '¿Cuánto es 15% de 340?'},
    {'type': 'reasoning', 'content': 'Necesito calcular 0.15 * 340'},
    {'type': 'tool', 'content': 'calculator(0.15 * 340)'},
    {'type': 'observation', 'content': '51.0'},
    {'type': 'response', 'content': 'El 15% de 340 es 51'}
]

fig = visualize_agent_trace(example_trace)
fig.show()

## 7. Ejercicios

### 🟢 Ejercicio 1: Agregar Nueva Herramienta

Implementa una herramienta `string_reverser` que invierta strings.

In [ ]:
def ejercicio_1_nueva_herramienta():
    """
    Objetivo: Crear una nueva herramienta y agregarla al agente
    
    Instrucciones:
    1. Define una función que revierta strings
    2. Crea un SimpleTool con esa función
    3. Crea un agente con esa herramienta
    4. Pruébala
    """
    # TODO: Tu código aquí
    # def reverse_string(text: str) -> str:
    #     ...
    
    # reverser_tool = SimpleTool(...)
    
    # agent = SimpleAgent(tools=[reverser_tool])
    # result = agent.run("Invierte la palabra 'python'")
    
    pass

# Test
def test_ejercicio_1():
    try:
        ejercicio_1_nueva_herramienta()
        print("✅ ¡Correcto! Has creado una herramienta nueva.")
        return True
    except:
        print("❌ Pista: Crea una función simple que use [::-1] para invertir el string")
        return False

# test_ejercicio_1()

### 🟡 Ejercicio 2: Implementar Memoria

Modifica `SimpleAgent` para que recuerde interacciones previas.

In [ ]:
def ejercicio_2_agente_con_memoria():
    """
    Objetivo: Implementar memoria en el agente
    
    Instrucciones:
    1. Modifica SimpleAgent para guardar historial de conversaciones
    2. En _build_prompt, incluye historial previo
    3. Prueba con múltiples queries que referencien información previa
    
    Ejemplo:
    - Query 1: "Calcula 10 * 5"
    - Query 2: "Ahora suma 20 al resultado anterior"
    """
    # TODO: Tu código aquí
    # Pista: Agrega self.conversation_history = [] en __init__
    # En cada run(), agrega la query y respuesta al historial
    # En _build_prompt(), incluye el historial
    
    pass

# Este ejercicio es más abierto - experimenta!

### 🔴 Ejercicio 3: Multi-Tool Planning

Crea un agente que pueda usar múltiples herramientas en secuencia para resolver una tarea compleja.

In [ ]:
def ejercicio_3_multi_tool_agent():
    """
    Objetivo: Resolver una tarea que requiere múltiples herramientas
    
    Tarea: "Calcula cuántos días faltan para el 31 de diciembre y luego 
            multiplica ese número por 24 para saber las horas."
    
    Herramientas necesarias:
    - get_time: Para saber la fecha actual
    - calculator: Para restar fechas y multiplicar
    
    Instrucciones:
    1. El agente debe decidir usar get_time primero
    2. Luego calcular días restantes
    3. Luego multiplicar por 24
    4. Finalmente responder
    """
    # TODO: Tu código aquí
    # Pista: Necesitas mejorar el loop del agente para permitir múltiples iteraciones
    # Pista 2: El agente necesita poder decidir usar otra herramienta después de recibir observación
    
    pass

# Este ejercicio es avanzado - requiere investigación adicional sobre ReAct pattern
# Lo exploraremos en profundidad en el notebook 03

## 8. Resumen y Recursos

### 📚 Resumen

En este notebook aprendiste:

- **LLM vs Agente**: Un LLM solo genera texto; un Agente usa LLMs para razonar + actuar
- **Componentes clave**: Cerebro (LLM), Memoria, Herramientas, Control Loop
- **Formalización matemática**: $\pi(s_t) = \text{LLM}(\text{prompt}(s_t))$
- **Implementación desde cero**: Loop básico observación → razonamiento → acción
- **Frameworks**: LangChain abstrae complejidad pero sacrifica control
- **Limitaciones actuales**: 
  - Costos de API pueden escalar rápidamente
  - Necesita prompt engineering cuidadoso
  - Puede entrar en loops infinitos sin límites
  - Seguridad: ejecución de código generado requiere sandboxing

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

1. **"ReAct: Synergizing Reasoning and Acting in Language Models"** (Yao et al., 2022)
   - [https://arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)
   - Contexto: Introduce el patrón que exploraremos en el notebook 03
   - Por qué es importante: Muestra cómo alternar razonamiento y acción mejora performance

2. **"Language Models as Agent Models"** (Andreas, 2022)
   - Análisis teórico de LLMs como agentes
   - Discute limitaciones y oportunidades

3. **"Toolformer: Language Models Can Teach Themselves to Use Tools"** (Schick et al., 2023)
   - [https://arxiv.org/abs/2302.04761](https://arxiv.org/abs/2302.04761)
   - Muestra cómo LLMs pueden aprender a usar herramientas

#### 🎥 Videos Recomendados

- **"Building LLM Agents"** - Harrison Chase (LangChain creator)
  - Overview de arquitecturas de agentes
  
- **"Agents 101"** - Andrew Ng (DeepLearning.AI)
  - Introducción conceptual accesible

#### 💻 Implementaciones de Referencia

- **LangChain**: [https://github.com/langchain-ai/langchain](https://github.com/langchain-ai/langchain)
  - Framework completo, bien documentado
  
- **AutoGPT**: [https://github.com/Significant-Gravitas/AutoGPT](https://github.com/Significant-Gravitas/AutoGPT)
  - Agente autónomo complejo
  
- **BabyAGI**: [https://github.com/yoheinakajima/babyagi](https://github.com/yoheinakajima/babyagi)
  - Implementación simple y educativa

#### 📖 Documentación

- **OpenAI Function Calling**: [https://platform.openai.com/docs/guides/function-calling](https://platform.openai.com/docs/guides/function-calling)
- **Anthropic Tool Use**: [https://docs.anthropic.com/claude/docs/tool-use](https://docs.anthropic.com/claude/docs/tool-use)
- **LangChain Agents**: [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)

### ➡️ Próximo Paso

En el siguiente notebook **"02. Prompting Agéntico"**, aprenderás técnicas avanzadas de prompting que hacen que los agentes razonen mejor:

- **Chain-of-Thought (CoT)**: Razonamiento paso a paso
- **Tree-of-Thought (ToT)**: Exploración de múltiples caminos
- **Self-Consistency**: Votación entre múltiples razonamientos
- **Few-shot prompting**: Aprender de ejemplos

Estas técnicas son fundamentales para construir agentes efectivos.

**[➡️ Ir al Notebook 02: Prompting Agéntico](02-prompting-agentico.ipynb)**

---

<div align="center">

### Respuesta a la Pregunta Guía

*¿Cómo transformamos un LLM pasivo en un agente activo?*

**Respuesta:** Envolvemos el LLM en un **control loop** que:
1. Construye prompts con contexto sobre herramientas disponibles
2. Parsea respuestas del LLM para identificar acciones
3. Ejecuta acciones en el entorno (usando herramientas)
4. Retroalimenta observaciones al LLM
5. Itera hasta completar la tarea

La "agenticidad" emerge de este loop, no del LLM en sí.

</div>